# File 01/3 – Morphology

### DESCRIPTION

This file provides an overview of all values occurring in the "morphology" attribute
in the PROIEL XML files.

The "morphology" attribute consists of positional morphosyntactic tags.
Each tag is a string of exactly 10 characters, where each position encodes a
specific morphological feature.

For example:
- Position 0 encodes person (1 = first, 2 = second, 3 = third)
- Position 1 encodes number (s = singular, d = dual, etc.)
- Subsequent positions encode additional grammatical features

Since different parts of speech encode different sets of features,
the placeholder "-" is used to maintain positional consistency across tags.

See the output for a complete overview of observed values.

#### INPUT FILES:

../source_data/treebank-releases-20180919

#### OUTPUT FILES:

None

## Imports

In [1]:
import os
import glob
import xml.etree.ElementTree as ET
from collections import defaultdict

## Input directory

In [2]:
# path to directory containing PROIEL XML files
folder = "../source_data/treebank-releases-20180919"

## Collect morphological mappings from XML files

In [3]:
# collect all XML files in the input directory
xml_files = glob.glob(os.path.join(folder, "*.xml"))

# dictionary mapping morphological fields (e.g. "person", "number") to their possible values
# structure: {field_tag: {code: description}}
# example: "number" -> {'s': 'singular', 'd': 'dual', ...}
morphology_dict = defaultdict(dict)

for xml_file in xml_files:
    try:
        # parse XML file and obtain root element
        tree = ET.parse(xml_file)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"Error parsing XML file '{xml_file}': {e}")
        continue

    # locate <annotation> element and its nested <morphology> block
    annotation = root.find('annotation')
    if annotation is None:
        continue

    morphology = annotation.find('morphology')
    if morphology is None:
        continue

    # iterate over <field> elements defining morphological categories
    for field in morphology.findall('field'):
        field_tag = field.get('tag')
        if not field_tag:
            continue

        # iterate over <value> elements representing possible values of the field
        for value in field.findall('value'):
            code = value.get('tag')
            summary = value.get('summary')

            if code and summary:
                # check for conflicting mappings
                if code in morphology_dict[field_tag]:
                    if morphology_dict[field_tag][code] != summary:
                        print(
                            f"Warning: conflicting values in field '{field_tag}' "
                            f"for code '{code}': "
                            f"'{morphology_dict[field_tag][code]}' vs. '{summary}'"
                        )
                else:
                    # store mapping from code to human-readable description
                    morphology_dict[field_tag][code] = summary

# print collected morphological mappings
print("Collected morphology data:")
for idx, (field, mapping) in enumerate(morphology_dict.items()):
    print(f"{idx} – {field}:\n {mapping})\n")

Collected morphology data:
0 – person:
 {'1': 'first person', '2': 'second person', '3': 'third person', 'x': 'uncertain person'})

1 – number:
 {'s': 'singular', 'd': 'dual', 'p': 'plural', 'x': 'uncertain number'})

2 – tense:
 {'p': 'present', 'i': 'imperfect', 'r': 'perfect', 's': 'resultative', 'a': 'aorist', 'u': 'past', 'l': 'pluperfect', 'f': 'future', 't': 'future perfect', 'x': 'uncertain tense'})

3 – mood:
 {'i': 'indicative', 's': 'subjunctive', 'm': 'imperative', 'o': 'optative', 'n': 'infinitive', 'p': 'participle', 'd': 'gerund', 'g': 'gerundive', 'u': 'supine', 'x': 'uncertain mood', 'y': 'finiteness unspecified', 'e': 'indicative or subjunctive', 'f': 'indicative or imperative', 'h': 'subjunctive or imperative', 't': 'finite'})

4 – voice:
 {'a': 'active', 'm': 'middle', 'p': 'passive', 'e': 'middle or passive', 'x': 'unspecified'})

5 – gender:
 {'m': 'masculine', 'f': 'feminine', 'n': 'neuter', 'p': 'masculine or feminine', 'o': 'masculine or neuter', 'r': 'feminine